# Asistente inteligente de proyectos sociales — RAG local

Consulta tres documentos PDF mediante embeddings `bge-m3`, ChromaDB y Granite 4 en Ollama. La interfaz está desarrollada con Gradio.

**Requisitos:** Ollama en ejecución, modelos `bge-m3` y `granite4:latest` descargados, dependencias instaladas y carpeta `chroma_db` existente junto al notebook. **No se vuelve a indexar la base en cada ejecución.**

## 1. Configuración y conexión con la base existente

In [ ]:
from pathlib import Path
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
import gradio as gr

RUTA_BASE = Path("chroma_db")
if not RUTA_BASE.exists():
    raise FileNotFoundError("No se encontró chroma_db. Ejecutá el notebook desde TP_RAG y conservá la base original.")

embeddings = OllamaEmbeddings(model="bge-m3", base_url="http://localhost:11434")
base_vectorial = Chroma(
    collection_name="programas_sociales_bge_m3",
    embedding_function=embeddings,
    persist_directory=str(RUTA_BASE),
)
llm = ChatOllama(model="granite4:latest", temperature=0)

cantidad = base_vectorial._collection.count()
print(f"Fragmentos disponibles: {cantidad}")
if cantidad != 491:
    print("Aviso: el conteo difiere de los 491 fragmentos de la versión evaluada.")

## 2. Consulta RAG y fuentes

Se recuperan tres fragmentos semánticamente relevantes. Los fragmentos mostrados son **contexto recuperado**, no una verificación automática de todas las afirmaciones de la respuesta.

In [ ]:
def consultar_rag_con_fuentes(pregunta):
    if not pregunta or not pregunta.strip():
        return "Escribí una pregunta para comenzar.", ""

    vector = embeddings.embed_query(pregunta.strip())
    resultados = base_vectorial._collection.query(
        query_embeddings=[vector],
        n_results=3,
        include=["documents", "metadatas"],
    )
    documentos = resultados["documents"][0]
    metadatos = resultados["metadatas"][0]
    if not documentos:
        return "No encontré esa información en los documentos.", ""

    contexto = "\n\n".join(
        f"Fragmento {i} (documento: {meta.get('fuente', 'Sin identificar')}, "
        f"página: {meta.get('pagina', 'Sin identificar')}):\n{texto}"
        for i, (texto, meta) in enumerate(zip(documentos, metadatos), start=1)
    )
    prompt = f"""Respondé en español, de forma clara y breve, usando solamente el contexto.
No inventes datos ni transformes objetivos institucionales en instrucciones personales.
Si el contexto no responde la pregunta, decí exactamente:
No encontré esa información en los documentos.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:"""
    respuesta = llm.invoke(prompt)

    fuentes = "### Fragmentos recuperados\n\n"
    for i, (texto, meta) in enumerate(zip(documentos, metadatos), start=1):
        cita = "\n> ".join(texto.splitlines())
        fuentes += (
            f"**Fragmento {i}** — **Documento:** {meta.get('fuente', 'Sin identificar')} "
            f"— **Página:** {meta.get('pagina', 'Sin identificar')}\n\n"
            f"> {cita}\n\n---\n\n"
        )
    return respuesta.content, fuentes

## 3. Interfaz Gradio

In [ ]:
with gr.Blocks(title="Asistente Inteligente de Proyectos Sociales") as demo_final:
    gr.Markdown("""# 📚 Asistente Inteligente de Proyectos Sociales
### Consultá información sobre convocatorias, servicios sociales y violencia de género
Respuestas basadas en tres documentos PDF mediante un sistema RAG local.
""")
    pregunta = gr.Textbox(label="Escribí tu pregunta", lines=2,
        placeholder="¿Cuántos proyectos puede presentar una organización?")
    boton = gr.Button("🔎 Consultar", variant="primary")
    respuesta = gr.Textbox(label="Respuesta del asistente", lines=6, interactive=False)
    fuentes = gr.Markdown()
    boton.click(fn=consultar_rag_con_fuentes, inputs=pregunta, outputs=[respuesta, fuentes])
    pregunta.submit(fn=consultar_rag_con_fuentes, inputs=pregunta, outputs=[respuesta, fuentes])

demo_final.launch(inbrowser=True)

## 4. Pruebas realizadas en la versión de desarrollo

| Tipo | Pregunta | Observación |
|---|---|---|
| Respondible — Convocatoria | ¿Cuántos proyectos sociales puede presentar una organización? | Respondió: hasta 3 proyectos |
| Respondible — Violencia de género | ¿Qué servicios de asesoramiento y contención ofrece la Línea 144? | Respuesta pertinente; verificar detalles con los fragmentos |
| Respondible — Servicios sociales | ¿Qué servicios ofrece la Ciudad de Buenos Aires para personas en situación de calle? | Identificó CIS, Red de Atención y Línea de Atención Social Inmediata; verificar teléfono |
| Sin respuesta documental | ¿Quién ganó el Mundial de Fútbol de 2022? | Indicó que no encontró información |
| Sin respuesta documental | ¿Cuál es el precio actual del dólar en Argentina? | Indicó que no encontró información |

**Limitaciones:** la generación local por CPU presentó demoras de decenas de segundos o más y algunas respuestas demasiado generales. Las pruebas anteriores se hicieron con el notebook de desarrollo; **esta versión limpia todavía debe validarse** antes de entregar.